In [ ]:
# Treatment-Focused Cytokine Analysis

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
import os
from datetime import datetime
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

def create_output_directory():
    """Create timestamped output directory for results"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"/Users/adityaelayavalli/Downloads/treatment_focused_analysis_{timestamp}"
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(f"{output_dir}/plots", exist_ok=True)
    os.makedirs(f"{output_dir}/data", exist_ok=True)
    os.makedirs(f"{output_dir}/statistics", exist_ok=True)
    print(f"Created output directory: {output_dir}")
    return output_dir

OUTPUT_DIR = None

def set_output_dir(output_dir):
    global OUTPUT_DIR
    OUTPUT_DIR = output_dir

def load_treatment_data(expression_csv='/Users/adityaelayavalli/Downloads/cytokine_panel_for_analysis.csv',
                       log_csv='/Users/adityaelayavalli/Downloads/GTBH25-SimonM-54_RNA_Project_Log_v01-25.csv'):
    """Load expression data and merge with treatment information"""
    
    # Load expression data
    expression_df = pd.read_csv(expression_csv, index_col=0)
    print(f"Loaded {expression_df.shape[0]} genes and {expression_df.shape[1]} samples")
    
    # Load treatment log
    log_df = pd.read_csv(log_csv)
    print(f"Loaded treatment information for {len(log_df)} samples")
    
    # Create mapping from GT numbers to treatment info
    sample_metadata = []
    
    for col in expression_df.columns:
        if col == 'Undetermined_S0_L007':
            continue  # Skip undetermined samples
            
        # Extract GT number from column name
        import re
        match = re.search(r'GT25\.(\d+)', col)
        if match:
            gt_number = f"GT25-{match.group(1)}"
            
            # Find matching row in log data
            log_row = log_df[log_df['Sample ID'] == gt_number]
            
            if not log_row.empty:
                log_info = log_row.iloc[0]
                
                # Extract animal ID and tissue from Customer ID
                customer_match = re.match(r'527_(\d+)_([^_]+)_RNA', log_info['Customer ID'])
                if customer_match:
                    animal_id, tissue_code = customer_match.groups()
                    
                    sample_metadata.append({
                        'sample_id': col,
                        'gt_number': gt_number,
                        'animal_id': animal_id,
                        'tissue_code': tissue_code,
                        'tissue': log_info['Tissue'],
                        'treatment': log_info['Treatment']
                    })
    
    # Create metadata DataFrame
    metadata_df = pd.DataFrame(sample_metadata)
    
    # Filter expression data to only include samples with metadata
    valid_samples = metadata_df['sample_id'].tolist()
    expression_df = expression_df[valid_samples]
    
    return expression_df, metadata_df

def filter_and_transform_data(expression_df, min_count=5, min_samples=3):
    """Filter low expression genes and log transform"""
    
    print(f"\nFiltering genes...")
    # Filter genes with low expression
    expressed_mask = (expression_df >= min_count).sum(axis=1) >= min_samples
    filtered_df = expression_df[expressed_mask]
    print(f"Kept {filtered_df.shape[0]} out of {expression_df.shape[0]} genes")
    
    # Log transform
    log_data = np.log2(filtered_df + 1)
    
    if OUTPUT_DIR:
        filtered_df.to_csv(f"{OUTPUT_DIR}/data/filtered_expression_data.csv")
        log_data.to_csv(f"{OUTPUT_DIR}/data/log_transformed_data.csv")
    
    return log_data

def analyze_treatment_effects_by_tissue(log_data, metadata_df, tissue_name):
    """Get treatment effects within a specific tissue (pooling genotypes)"""
    
    print(f"\n=== Analyzing {tissue_name} ===")
    
    # Filter data for this tissue
    tissue_mask = metadata_df['tissue'] == tissue_name
    tissue_metadata = metadata_df[tissue_mask].copy()
    tissue_expression = log_data[tissue_metadata['sample_id']]
    
    print(f"Total samples in {tissue_name}: {len(tissue_metadata)}")
    
    # Treatment summary for this tissue
    treatment_summary = tissue_metadata.groupby('treatment').size().reset_index(name='count')
    print(f"Treatment distribution:")
    for _, row in treatment_summary.iterrows():
        print(f"  {row['treatment']}: {row['count']} samples")
    
    results = {
        'tissue': tissue_name,
        'metadata': tissue_metadata,
        'expression': tissue_expression,
        'treatment_summary': treatment_summary
    }
    
    return results

def perform_treatment_pca(tissue_data, tissue_name):
    """Perform PCA analysis colored by treatment"""
    
    expression = tissue_data['expression']
    metadata = tissue_data['metadata']
    
    # Transpose for PCA (samples as rows, genes as columns)
    data_for_pca = expression.T
    
    # Perform PCA
    pca = PCA()
    pca_result = pca.fit_transform(data_for_pca)
    explained_variance = pca.explained_variance_ratio_ * 100
    
    # Create PCA DataFrame
    pca_df = pd.DataFrame({
        'PC1': pca_result[:, 0],
        'PC2': pca_result[:, 1] if pca_result.shape[1] > 1 else np.zeros(len(pca_result)),
        'PC3': pca_result[:, 2] if pca_result.shape[1] > 2 else np.zeros(len(pca_result)),
        'sample_id': metadata['sample_id'].values,
        'animal_id': metadata['animal_id'].values,
        'treatment': metadata['treatment'].values
    })
    
    # Create PCA plots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # PC1 vs PC2 colored by treatment
    treatments = pca_df['treatment'].unique()
    colors_treatment = plt.cm.Set1(np.linspace(0, 1, len(treatments)))
    
    for i, treatment in enumerate(treatments):
        mask = pca_df['treatment'] == treatment
        axes[0,0].scatter(pca_df.loc[mask, 'PC1'], pca_df.loc[mask, 'PC2'],
                         c=[colors_treatment[i]], label=treatment, alpha=0.7, s=100)
    
    axes[0,0].set_xlabel(f'PC1: {explained_variance[0]:.1f}% variance')
    axes[0,0].set_ylabel(f'PC2: {explained_variance[1]:.1f}% variance' if len(explained_variance) > 1 else 'PC2')
    axes[0,0].legend()
    axes[0,0].set_title(f'{tissue_name}: PCA by Treatment')
    axes[0,0].grid(True, alpha=0.3)
    
    # PC1 vs PC2 colored by animal ID
    animals = pca_df['animal_id'].unique()
    colors_animal = plt.cm.tab20(np.linspace(0, 1, len(animals)))
    
    for i, animal in enumerate(animals):
        mask = pca_df['animal_id'] == animal
        if mask.sum() > 0:
            axes[0,1].scatter(pca_df.loc[mask, 'PC1'], pca_df.loc[mask, 'PC2'],
                            c=[colors_animal[i]], label=f'Animal {animal}', alpha=0.7, s=100)
    
    axes[0,1].set_xlabel(f'PC1: {explained_variance[0]:.1f}% variance')
    axes[0,1].set_ylabel(f'PC2: {explained_variance[1]:.1f}% variance' if len(explained_variance) > 1 else 'PC2')
    axes[0,1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    axes[0,1].set_title(f'{tissue_name}: PCA by Animal')
    axes[0,1].grid(True, alpha=0.3)
    
    # PC3 vs PC4 by treatment (if available)
    if pca_result.shape[1] > 2:
        for i, treatment in enumerate(treatments):
            mask = pca_df['treatment'] == treatment
            pc3_data = pca_df.loc[mask, 'PC3']
            # Only use PC4 if it exists, otherwise use zeros
            if pca_result.shape[1] > 3:
                pc4_data = pca_result[mask, 3]
            else:
                pc4_data = np.zeros(mask.sum())
            
            axes[1,0].scatter(pc3_data, pc4_data,
                             c=[colors_treatment[i]], label=treatment, alpha=0.7, s=100)
        
        axes[1,0].set_xlabel(f'PC3: {explained_variance[2]:.1f}% variance' if len(explained_variance) > 2 else 'PC3')
        axes[1,0].set_ylabel(f'PC4: {explained_variance[3]:.1f}% variance' if len(explained_variance) > 3 else 'PC4 (not available)')
        axes[1,0].legend()
        axes[1,0].set_title(f'{tissue_name}: PC3 vs PC4')
    else:
        axes[1,0].text(0.5, 0.5, 'PC3/PC4 not available\n(insufficient components)', 
                      ha='center', va='center', transform=axes[1,0].transAxes, fontsize=12)
        axes[1,0].set_title(f'{tissue_name}: PC3 vs PC4 (Not Available)')
    axes[1,0].grid(True, alpha=0.3)
    
    # Variance explained plot
    n_pcs_to_plot = min(10, len(explained_variance))
    axes[1,1].plot(range(1, n_pcs_to_plot + 1), explained_variance[:n_pcs_to_plot], 'o-')
    axes[1,1].set_xlabel('Principal Component')
    axes[1,1].set_ylabel('Explained Variance (%)')
    axes[1,1].set_title(f'{tissue_name}: Variance Explained')
    axes[1,1].grid(True, alpha=0.3)
    
    plt.suptitle(f'{tissue_name} Treatment PCA Analysis', fontsize=16)
    plt.tight_layout()
    
    if OUTPUT_DIR:
        plt.savefig(f"{OUTPUT_DIR}/plots/{tissue_name.replace(' ', '_')}_treatment_pca.png", 
                   dpi=300, bbox_inches='tight')
        plt.savefig(f"{OUTPUT_DIR}/plots/{tissue_name.replace(' ', '_')}_treatment_pca.pdf", 
                   bbox_inches='tight')
    
    plt.show()
    
    return pca, pca_result, pca_df

def compare_treatments_pairwise(tissue_data, tissue_name, treatment1, treatment2):
    """Compare two treatments directly"""
    
    expression = tissue_data['expression']
    metadata = tissue_data['metadata']
    
    print(f"\n--- {tissue_name}: {treatment1} vs {treatment2} ---")
    
    # Get samples for each treatment
    treat1_samples = metadata[metadata['treatment'] == treatment1]['sample_id']
    treat2_samples = metadata[metadata['treatment'] == treatment2]['sample_id']
    
    if len(treat1_samples) == 0 or len(treat2_samples) == 0:
        print(f"Insufficient samples - {treatment1}: {len(treat1_samples)}, {treatment2}: {len(treat2_samples)}")
        return pd.DataFrame()
    
    print(f"{treatment1}: {len(treat1_samples)} samples, {treatment2}: {len(treat2_samples)} samples")
    
    results = []
    
    for gene in expression.index:
        treat1_expr = expression.loc[gene, treat1_samples]
        treat2_expr = expression.loc[gene, treat2_samples]
        
        mean_treat1 = treat1_expr.mean()
        mean_treat2 = treat2_expr.mean()
        log2_fc = mean_treat1 - mean_treat2  # treatment1 vs treatment2
        
        # t-test
        try:
            statistic, pvalue = stats.ttest_ind(treat1_expr, treat2_expr, equal_var=False)
        except:
            pvalue = 1.0
            statistic = 0.0
        
        results.append({
            'tissue': tissue_name,
            'comparison': f'{treatment1}_vs_{treatment2}',
            'gene': gene,
            'log2FoldChange': log2_fc,
            'pvalue': pvalue,
            f'{treatment1}_mean': mean_treat1,
            f'{treatment2}_mean': mean_treat2,
            'effect_size': abs(log2_fc)
        })
    
    # Convert to DataFrame and adjust p-values
    de_results = pd.DataFrame(results)
    
    # Adjust p-values
    if len(de_results) > 0:
        _, padj, _, _ = multipletests(de_results['pvalue'], method='fdr_bh')
        de_results['padj'] = padj
        
        # Sort by effect size
        de_results = de_results.sort_values('effect_size', ascending=False)
        
        # Save results
        if OUTPUT_DIR:
            filename = f"{tissue_name.replace(' ', '_')}_{treatment1}_vs_{treatment2}_DE.csv"
            de_results.to_csv(f"{OUTPUT_DIR}/statistics/{filename}", index=False)
    
    return de_results

def plot_treatment_volcano(de_results, tissue_name, comparison_name, pval_cutoff=0.8, fc_cutoff=0.05):
    """Create volcano plot for treatment comparisons - shows top genes regardless of significance"""
    
    if len(de_results) == 0:
        print(f"No data to plot for {comparison_name}")
        return
    
    plt.figure(figsize=(12, 8))
    
    # Color points by significance and direction
    colors = []
    significant_genes = []
    
    for _, row in de_results.iterrows():
        if row['padj'] < pval_cutoff and abs(row['log2FoldChange']) > fc_cutoff:
            if row['log2FoldChange'] > 0:
                colors.append('red')
                significant_genes.append((row['gene'], row['log2FoldChange'], -np.log10(row['padj'])))
            else:
                colors.append('blue')
                significant_genes.append((row['gene'], row['log2FoldChange'], -np.log10(row['padj'])))
        else:
            colors.append('gray')
    
    # If no significant genes, show top genes by effect size
    if len(significant_genes) == 0:
        print(f"No significant genes found. Showing top genes by effect size.")
        top_genes = de_results.nlargest(8, 'effect_size')
        for _, row in top_genes.iterrows():
            if row['log2FoldChange'] > 0:
                significant_genes.append((row['gene'], row['log2FoldChange'], -np.log10(row['padj'])))
            else:
                significant_genes.append((row['gene'], row['log2FoldChange'], -np.log10(row['padj'])))
    
    # If very few significant genes, show top genes by effect size regardless
    if len(significant_genes) < 3:
        print(f"Very few significant genes. Showing top genes by effect size.")
        top_genes = de_results.nlargest(8, 'effect_size')
        significant_genes = []
        for _, row in top_genes.iterrows():
            significant_genes.append((row['gene'], row['log2FoldChange'], -np.log10(row['padj'])))
    
    # Create scatter plot
    scatter = plt.scatter(de_results['log2FoldChange'], -np.log10(de_results['padj']), 
                         c=colors, alpha=0.7, s=80)
    
    # Add gene labels for significant or top genes
    significant_genes.sort(key=lambda x: x[2], reverse=True)  # Sort by -log10(padj)
    
    for gene, fc, neg_log_p in significant_genes[:8]:  # Top 8 genes
        plt.annotate(gene, (fc, neg_log_p),
                    xytext=(5, 5), textcoords='offset points', fontsize=10,
                    ha='left', alpha=0.8, fontweight='bold')
    
    # Add cutoff lines
    plt.axhline(y=-np.log10(pval_cutoff), color='black', linestyle='--', alpha=0.5, label=f'p = {pval_cutoff}')
    plt.axvline(x=fc_cutoff, color='black', linestyle='--', alpha=0.5)
    plt.axvline(x=-fc_cutoff, color='black', linestyle='--', alpha=0.5)
    
    plt.xlabel(f'Log2 Fold Change')
    plt.ylabel('-Log10 Adjusted P-value')
    plt.title(f'{tissue_name}: {comparison_name}')
    plt.grid(True, alpha=0.3)
    
    # Add legend
    from matplotlib.patches import Patch
    treatments = comparison_name.split(' vs ')
    legend_elements = [
        Patch(facecolor='red', label=f'Higher in {treatments[0]}'),
        Patch(facecolor='blue', label=f'Higher in {treatments[1]}'),
        Patch(facecolor='gray', label='Not significant')
    ]
    plt.legend(handles=legend_elements)
    
    plt.tight_layout()
    
    # Save plot
    if OUTPUT_DIR:
        safe_name = comparison_name.replace(' ', '_').replace('vs', 'vs')
        plt.savefig(f"{OUTPUT_DIR}/plots/{tissue_name.replace(' ', '_')}_{safe_name}_volcano.png", 
                   dpi=300, bbox_inches='tight')
        plt.savefig(f"{OUTPUT_DIR}/plots/{tissue_name.replace(' ', '_')}_{safe_name}_volcano.pdf", 
                   bbox_inches='tight')
    
    plt.show()
    
    # Print summary
    significant_count = len([g for g in significant_genes if g[2] > -np.log10(pval_cutoff)])
    print(f"\n{comparison_name} Summary:")
    print(f"Total genes: {len(de_results)}")
    print(f"Significant genes (padj < {pval_cutoff}, |FC| > {fc_cutoff}): {significant_count}")
    
    if len(significant_genes) > 0:
        print("Top genes by effect size:")
        for gene, fc, neg_log_p in significant_genes[:5]:
            print(f"  {gene}: FC={fc:.2f}, padj={10**(-neg_log_p):.2e}")
    else:
        print("No genes meet significance criteria - consider biological relevance over statistical significance")

def plot_treatment_heatmap(tissue_data, tissue_name):
    """Create heatmap showing cytokine expression across treatments"""
    
    expression = tissue_data['expression']
    metadata = tissue_data['metadata']
    
    # Calculate mean expression for each treatment
    conditions = []
    condition_labels = []
    
    for treatment in sorted(metadata['treatment'].unique()):
        samples = metadata[metadata['treatment'] == treatment]['sample_id']
        
        if len(samples) > 0:
            mean_expr = expression[samples].mean(axis=1)
            conditions.append(mean_expr)
            condition_labels.append(treatment)
    
    # Create heatmap data
    heatmap_data = pd.DataFrame(conditions).T
    heatmap_data.columns = condition_labels
    
    # Plot heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(heatmap_data, annot=True, cmap='viridis', fmt='.1f',
                cbar_kws={'label': 'Mean Expression (log2)'})
    plt.title(f'{tissue_name}: Cytokine Expression by Treatment')
    plt.ylabel('Cytokines')
    plt.xlabel('Treatment')
    
    plt.tight_layout()
    
    # Save plot
    if OUTPUT_DIR:
        plt.savefig(f"{OUTPUT_DIR}/plots/{tissue_name.replace(' ', '_')}_treatment_heatmap.png", 
                   dpi=300, bbox_inches='tight')
        plt.savefig(f"{OUTPUT_DIR}/plots/{tissue_name.replace(' ', '_')}_treatment_heatmap.pdf", 
                   bbox_inches='tight')
    
    plt.show()
    
    # Save heatmap data
    if OUTPUT_DIR:
        heatmap_data.to_csv(f"{OUTPUT_DIR}/data/{tissue_name.replace(' ', '_')}_treatment_heatmap_data.csv")
    
    return heatmap_data

def run_treatment_focused_analysis():
    """Run complete treatment-focused analysis"""
    
    # Create output directory
    output_dir = create_output_directory()
    set_output_dir(output_dir)
    
    print("=== TREATMENT-FOCUSED CYTOKINE ANALYSIS ===")
    print("Comparing treatments across tissues")
    print("Focus: Drug effects on cytokine expression")
    
    # 1. Load data
    print("\n1. Loading data...")
    expression_data, metadata = load_treatment_data()
    
    # 2. Filter and transform data
    print("\n2. Filtering and transforming data...")
    log_data = filter_and_transform_data(expression_data)
    
    print(f"Final dataset: {log_data.shape[0]} cytokines, {log_data.shape[1]} samples")
    print(f"Cytokines analyzed: {', '.join(log_data.index)}")
    
    # 3. Analyze each tissue
    tissues = sorted(metadata['tissue'].unique())
    treatments = sorted(metadata['treatment'].unique())
    
    all_results = {}
    
    for tissue in tissues:
        print(f"\n{'='*60}")
        print(f"ANALYZING {tissue.upper()}")
        print(f"{'='*60}")
        
        # Get tissue-specific data
        tissue_data = analyze_treatment_effects_by_tissue(log_data, metadata, tissue)
        
        # PCA analysis
        print(f"\n3.1. PCA Analysis for {tissue}")
        pca, pca_result, pca_df = perform_treatment_pca(tissue_data, tissue)
        
        # Treatment heatmap
        print(f"\n3.2. Expression Heatmap for {tissue}")
        heatmap_data = plot_treatment_heatmap(tissue_data, tissue)
        
        # Pairwise treatment comparisons
        print(f"\n3.3. Pairwise Treatment Comparisons for {tissue}")
        
        de_results = {}
        
        # All pairwise comparisons
        for i, treat1 in enumerate(treatments):
            for treat2 in treatments[i+1:]:
                comparison_name = f"{treat1} vs {treat2}"
                de_result = compare_treatments_pairwise(tissue_data, tissue, treat1, treat2)
                
                if len(de_result) > 0:
                    plot_treatment_volcano(de_result, tissue, comparison_name)
                    de_results[f"{treat1}_vs_{treat2}"] = de_result
        
        # Store results
        all_results[tissue] = {
            'tissue_data': tissue_data,
            'pca_results': pca_df,
            'heatmap_data': heatmap_data,
            'de_results': de_results
        }
    
    # 4. Cross-tissue summary
    create_cross_tissue_summary(all_results, treatments, output_dir)
    
    print(f"TREATMENT-FOCUSED ANALYSIS COMPLETE!")
    print(f"All results saved to: {output_dir}")
    print(f"Focus: Treatment effects on cytokine expression")
    print(f"Check volcano plots for cytokine changes between treatments")
    
    return all_results

def create_cross_tissue_summary(all_results, treatments, output_dir):
    """Create cross-tissue treatment effect summary"""
    
    print(f"\n4. Cross-tissue treatment effect summary...")
    
    # Compile all significant effects
    summary_data = []
    
    for tissue_name, tissue_results in all_results.items():
        if 'de_results' in tissue_results:
            for comparison, de_data in tissue_results['de_results'].items():
                if len(de_data) > 0:
                    # Count significant genes
                    significant = de_data[
                        (de_data['padj'] < 0.1) & 
                        (abs(de_data['log2FoldChange']) > 0.3)
                    ]
                    
                    summary_data.append({
                        'tissue': tissue_name,
                        'comparison': comparison.replace('_vs_', ' vs '),
                        'total_genes': len(de_data),
                        'significant_genes': len(significant),
                        'percent_significant': len(significant) / len(de_data) * 100 if len(de_data) > 0 else 0,
                        'max_effect_size': de_data['effect_size'].max() if len(de_data) > 0 else 0
                    })
    
    if len(summary_data) > 0:
        summary_df = pd.DataFrame(summary_data)
        
        # Save summary
        if output_dir:
            summary_df.to_csv(f"{output_dir}/statistics/treatment_effects_summary.csv", index=False)
        
        # Plot summary
        plt.figure(figsize=(15, 8))
        
        # Pivot for heatmap
        pivot_data = summary_df.pivot(index='comparison', columns='tissue', values='percent_significant')
        
        sns.heatmap(pivot_data, annot=True, cmap='YlOrRd', fmt='.1f',
                    cbar_kws={'label': 'Percent Significant Genes'})
        plt.title('Treatment Effects Across Tissues\n(% of genes significantly changed)')
        plt.xlabel('Tissue')
        plt.ylabel('Treatment Comparison')
        plt.xticks(rotation=45)
        plt.yticks(rotation=0)
        
        plt.tight_layout()
        
        if output_dir:
            plt.savefig(f"{output_dir}/plots/cross_tissue_treatment_summary.png", dpi=300, bbox_inches='tight')
            plt.savefig(f"{output_dir}/plots/cross_tissue_treatment_summary.pdf", bbox_inches='tight')
        
        plt.show()
        
        # Print key findings
        print("KEY FINDINGS:")
        print("=" * 40)
        
        # Most responsive tissue-treatment combinations
        top_effects = summary_df.nlargest(5, 'percent_significant')
        print("Most responsive tissue-treatment combinations:")
        for _, row in top_effects.iterrows():
            print(f"  {row['tissue']} - {row['comparison']}: {row['percent_significant']:.1f}% significant")
        
        # Treatment with most effects overall
        treatment_effects = summary_df.groupby('comparison')['significant_genes'].sum().sort_values(ascending=False)
        print(f"\nTreatment comparisons with most effects overall:")
        for comparison, count in treatment_effects.head(3).items():
            print(f"  {comparison}: {count} significant genes across all tissues")

# Execute the analysis
if __name__ == "__main__":
    print("Starting Treatment-Focused Cytokine Analysis...")
    print("Comparing drug effects across tissues")
    
    results = run_treatment_focused_analysis()
    
    print("Analysis complete! Pure treatment comparison approach.")